# Follow the Workers

Staged, auditable research notebook. Use `/opt/homebrew/bin/python3` and the project directory as the working directory. Research selection ends September 2014; October is purged; sealed returns run November 2014–August 2026. The notebook stops before experimental calls unless their transport has been verified. No cell silently opens the seal.

In [ ]:
import sys, json, inspect, importlib.metadata
from pathlib import Path
import pandas as pd
import params as P
import data as D
import stats as S
import agents as A
import plots
assert sys.version_info >= (3,11), "Use the recorded Homebrew Python runtime"
assert Path.cwd().resolve() == P.ROOT, "Run with the notebook folder as the working directory"
def checkpoint(stage, message):
    path = P.OUT / "checkpoints.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as stream:
        stream.write(f"\n- {D.utc_now()} · {stage}: {message}\n")


## Stage 0: preserve the approved design

Create or verify the preregistration before any numeric return parser runs. A changed parameter file requires an explicit, recorded amendment; nothing is overwritten.

In [ ]:
prereg = D.stage0()
checkpoint("Stage 0", "Preregistration verified; 125 selection months and 142 sealed months")
prereg

## Stage 1: pin and audit sources

Use the provided vintage archive and pinned official French downloads. Verify live first-vintage metadata with the configured FRED credential. The P1 packet requires a documented Opus 5.5 extra-high audit before Stage 2. Missing evidence stops the notebook.


In [ ]:
inventory = D.prepare_inputs()
live_vintages = D.fred_metadata_audit()
manifest = D.manifest()
crosswalk = D.crosswalk()
packet = P.P1_PROMPT + "\n\nManifest:\n```csv\n" + manifest.to_csv(index=False) + "```\nCrosswalk:\n```csv\n" + (P.OUT/"crosswalk.csv").read_text() + "```\nAs-of code:\n```python\n" + inspect.getsource(D.read_vintage) + inspect.getsource(D.vintage_view) + inspect.getsource(D.build_asof_panel) + "```\n"
(P.OUT/"p1_bundle.md").write_text(packet)
checkpoint("Stage 1", f"{len(inventory)} pinned inputs; optional company data excluded for documented historical-evidence failures")
print("Inputs pinned. Complete the P1 review before Stage 2.")

## Stages 2–3: point-in-time labor panels and features

Prepare four variants without parsing returns. Reuse existing artifacts; verify their source hashes, dates, fixed-lag bounds, complete group grid and feature coverage. These preparation checks do not substitute for P1.

In [ ]:
A.require_p1_audit()
for variant in P.VARIANTS:
    panel_path = P.OUT / f"panel_{variant.lower()}.parquet"
    feature_path = P.OUT / f"features_{variant.lower()}.parquet"
    if not panel_path.exists():
        D.build_asof_panel(variant)
    if not feature_path.exists():
        D.build_features(pd.read_parquet(panel_path))
panel_audit = D.audit_prepared_data()
checkpoint("Stages 2–3", "All four labor variants passed schema, calendar and publication-cutoff checks")
panel_audit["variants"]

## Stage 4: blind the experiment

The private map remains local. Agents receive only the anonymous dictionary and rounded research feedback. Raw levels and target arrays are evaluated locally and never sent. Export all 18 first prompts as the no-API fallback.

In [ ]:
features = pd.read_parquet(P.OUT/"features_asof.parquet")
levels = pd.read_parquet(P.OUT/"levels_asof.parquet")
tightness_table = pd.read_parquet(P.OUT/"tightness_asof.parquet")
variables, tightness, dictionary, blind = A.blind_inputs(levels, tightness_table)
variables = {key: value.loc[:P.SELECTION_END] for key,value in variables.items()}
blind_groups = {int(key):value for key,value in blind["groups"].items()}
base = S.feature_matrices(features)
base_blind = {key:value.rename(columns=blind_groups).sort_index(axis=1).loc[:P.SELECTION_END] for key,value in base.items()}
first_prompts = A.initial_prompts(dictionary)
checkpoint("Stage 4", "Private blinding map and 18 first prompts saved; no model calls made")
print(len(first_prompts), "initial prompts exported")

## Stage 6: test the evaluator before agent research

Run synthetic cutoff, portfolio, inference and parser checks. Load only research returns through September 2014 after preregistration. This is a data-integrity check, not feature selection or strategy evaluation.

In [ ]:
checks = {"passed":True, "data":D.self_test(), "stats":S.self_test(), "agents":A.self_test()}
checks["evidence_hashes"] = {name:D.sha256(P.ROOT/name) for name in ["params.py","data.py","stats.py","agents.py","plots.py"]}
D.write_json(P.OUT/"synthetic_checks.json", checks)
D.write_json(P.OUT/"runtime.json", {"executable":sys.executable,"python":sys.version.split()[0],"packages":{name:importlib.metadata.version(name) for name in ["pandas","numpy","scipy","statsmodels","scikit-learn","matplotlib","requests","pyarrow","nbformat"]}})
research_returns = D.read_returns()
target_blind = research_returns["relative"].rename(columns=blind_groups).sort_index(axis=1)
assert target_blind.index.max() == pd.Period(P.SELECTION_END)
plots.design_figures()
checkpoint("Stage 6", "Synthetic tests passed; research return endpoint/schema checked; sealed returns remain unopened")
print("Research endpoint:", str(target_blind.index.max()), "groups:", target_blind.shape[1])

## Stage 5: research calls and migration tracing

The approved transport is Claude Opus 5.5 extra-high through the existing subscription. This cell stops until the exact CLI/model and no-tools configuration are verified and P1 is complete. Each call starts a fresh isolated process; completed responses are cached. Run indices 1–3 are independent repetitions, not controllable model random seeds. Only run 1 can supply strategy features. No model fallback is configured.

In [ ]:
A.require_transport()
transport = A.claude_transport
specs, candidate_values_blind, migration_reports = A.run_experiment(variables, tightness.loc[:P.SELECTION_END], dictionary, target_blind, base_blind, transport=transport)
judgments = A.trace_migrations(specs, migration_reports, transport=transport)
A.summarize_search(specs)
A.summarize_migrations(specs, migration_reports, judgments)
A.audit_agent_prompts()
checkpoint("Stage 5", "Fixed-budget calls and tracing recorded; required human review remains explicit")

## Stage 7: mechanical selection, review and freeze

Only seed 1 supplies features. The human migration spot-check and P1/P4 records must be completed and bound to the current artifacts. P4 receives anonymous research tables, never sealed results. The capacity record is pinned even when the answer is unavailable.

In [ ]:
decode_groups = {value:int(key) for key,value in blind["groups"].items()}
candidate_values = {key:value.rename(columns=decode_groups).sort_index(axis=1) for key,value in candidate_values_blind.items()}
configuration, research_summary = S.research_run(base,tightness,specs,candidate_values,research_returns)
D.write_json(P.OUT/"p4_prompt.json", {"prompt":P.P4_PROMPT,"anonymous_strategy_tables":research_summary})
print(S.readiness())
# Complete P1/P4 evidence and the sampled human judgments, then rerun the summary.
# Freeze raises while any prerequisite is missing, stale or unverified.
freeze_record = S.freeze()
checkpoint("Stage 7", "Feature choices, data, code, audits and capacity record frozen")

## Stage 8: open the sealed test once

This is the only entry point for sealed numeric returns. A durable start marker is written before loading. A repeated execution requires a written reason and preserves a separate output directory.

In [ ]:
final_result = D.final_evaluation(S.evaluate_frozen)
checkpoint("Stage 8", "Initial sealed evaluation completed; all outputs saved under sealed/initial")

## Stage 9: figures and report

Report every registered result, including failed or unavailable diagnostics. Economic interpretation happens after selection is frozen. Students fill the team-critique column; ETF research proxies do not establish implementable capacity.

In [ ]:
plots.result_figures("initial")
report_path = plots.write_report("initial")
checkpoint("Stage 9", final_result["gate"]["verdict"] + "; see saved criteria, limitations, all diagnostics and reproducibility notebook")
print(report_path)